In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_val_score,
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer)

# ---------------------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------------------
CSV_PATH = "networkTraffic.csv"
TARGET_COLUMN = "attack_cat"
ID_COLUMN = "id"
CATEGORICAL_COLUMNS = ["proto", "state", "service"]
RANDOM_STATE = 42
TUNING_SUBSET_FRACTION = 0.25

# ---------------------------------------------------------------------
# 2. Load and Clean Data
# ---------------------------------------------------------------------
df = pd.read_csv(CSV_PATH, na_values="?")

if ID_COLUMN in df.columns:
    df = df.drop(columns=[ID_COLUMN])

df = df.dropna(subset=[TARGET_COLUMN])

X = df.drop(columns=[TARGET_COLUMN])

X = X.drop(columns=["sloss", "dloss", "ct_ftp_cmd"], errors="ignore")
y = df[TARGET_COLUMN]

# ---------------------------------------------------------------------
# 3. Identify numeric vs categorical descriptive features
# ---------------------------------------------------------------------
categorical_cols = [c for c in CATEGORICAL_COLUMNS if c in X.columns]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

X[numeric_cols] = X[numeric_cols].astype(np.float32)

# ---------------------------------------------------------------------
# 4. Preprocessing pipeline
# ---------------------------------------------------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

knn_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("knn", KNeighborsClassifier()),
])

scorer = make_scorer(f1_score, average="macro")
print("Preprocessing pipeline configured.")

# ---------------------------------------------------------------------
# 5. Train/test split
# ---------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# ---------------------------------------------------------------------
# 6. Hyperparameter tuning via GridSearchCV on a tuning SUBSET
# ---------------------------------------------------------------------

X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train,
    train_size=TUNING_SUBSET_FRACTION,
    stratify=y_train,
    random_state=RANDOM_STATE,
)

param_grid = {
    "knn__n_neighbors": [5, 10, 15, 20, 25],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"],
}

grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    scoring=scorer,
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

print(f"Starting kNN grid search on {TUNING_SUBSET_FRACTION:.0%} tuning subset...")
grid_search.fit(X_tune, y_tune)

best_idx = grid_search.best_index_
best_mean = grid_search.cv_results_["mean_test_score"][best_idx]
best_std = grid_search.cv_results_["std_test_score"][best_idx]

print("Best hyperparameters:", grid_search.best_params_)
print(f"Best cross-validated macro-F1 (tuning subset): {best_mean:.4f} +/- {best_std:.4f} "
      f"(over {cv.get_n_splits()} folds, n={len(X_tune)})")

# ---------------------------------------------------------------------
# 7. Refit best configuration on the FULL training set, then evaluate
# ---------------------------------------------------------------------

best_model = grid_search.best_estimator_

full_cv_scores = cross_val_score(
    best_model, X_train, y_train, cv=cv, scoring=scorer, n_jobs=-1
)
print(f"\nCross-validated macro-F1 (full training set, same hyperparameters): "
      f"{full_cv_scores.mean():.4f} +/- {full_cv_scores.std():.4f} "
      f"(over {cv.get_n_splits()} folds, n={len(X_train)})")

print("\nFitting best model on full training set...")
best_model.fit(X_train, y_train)

print("Predicting on held-out test set...")
y_pred = best_model.predict(X_test)

print("\nClassification report on held-out test set:")
print(classification_report(y_test, y_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(10, 8), dpi=300)

disp = ConfusionMatrixDisplay.from_predictions(
    y_test, 
    y_pred, 
    ax=ax, 
    cmap="Blues",             
    xticks_rotation="vertical", 
    colorbar=True,
    values_format="d"        
)

plt.title("Confusion Matrix – UNSW-NB15 Classification", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Predicted Attack Category", fontsize=12, labelpad=10)
plt.ylabel("True Attack Category", fontsize=12, labelpad=10)
plt.grid(False)               
plt.tight_layout()

plt.savefig("confusion_matrix_plot.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer)

# ---------------------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------------------

CSV_PATH = "networkTraffic.csv"
TARGET_COLUMN = "attack_cat"
ID_COLUMN = "id"
CATEGORICAL_COLUMNS = ["proto", "state", "service"]
RANDOM_STATE = 42

# ---------------------------------------------------------------------
# 2. Load and Clean Data
# ---------------------------------------------------------------------
df = pd.read_csv(CSV_PATH, na_values="?")

if ID_COLUMN in df.columns:
    df = df.drop(columns=[ID_COLUMN])

df = df.dropna(subset=[TARGET_COLUMN])

X = df.drop(columns=[TARGET_COLUMN])
# Drop redundant features
X = X.drop(columns=["sloss", "dloss", "ct_ftp_cmd"], errors="ignore")
y = df[TARGET_COLUMN]

# ---------------------------------------------------------------------
# 3. Identify numeric vs categorical descriptive features
# ---------------------------------------------------------------------
categorical_cols = [c for c in CATEGORICAL_COLUMNS if c in X.columns]
numeric_cols = [c for c in X.columns if c not in categorical_cols]

# ---------------------------------------------------------------------
# 4. Preprocessing pipeline (No Scaler needed for Trees)
# ---------------------------------------------------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

tree_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("tree", DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

scorer = make_scorer(f1_score, average="macro")

# ---------------------------------------------------------------------
# 5. Train/test split
# ---------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# ---------------------------------------------------------------------
# 6. Hyperparameter tuning via GridSearchCV
# ---------------------------------------------------------------------
param_grid = {
    "tree__max_depth": [5, 10, 15, 20],
    "tree__min_samples_split": [5, 10, 15, 20],
    "tree__criterion": ["gini", "entropy"],
}

grid_search = GridSearchCV(
    estimator=tree_pipeline,
    param_grid=param_grid,
    scoring=scorer,
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

print("Starting Decision Tree grid search on full training set...")
grid_search.fit(X_train, y_train)

# Extract mean AND standard deviation for the best configuration
best_idx = grid_search.best_index_
best_mean = grid_search.cv_results_["mean_test_score"][best_idx]
best_std = grid_search.cv_results_["std_test_score"][best_idx]

print("Best hyperparameters:", grid_search.best_params_)
print(f"Best cross-validated macro-F1: {best_mean:.4f} +/- {best_std:.4f} "
      f"(over {cv.get_n_splits()} folds)")

# ---------------------------------------------------------------------
# 7. Final Model Evaluation
# ---------------------------------------------------------------------
best_tree_model = grid_search.best_estimator_

print("\nPredicting on held-out test set...")
y_pred = best_tree_model.predict(X_test)

print("\nClassification report on held-out test set:")
print(classification_report(y_test, y_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(10, 8), dpi=300)

disp = ConfusionMatrixDisplay.from_predictions(
    y_test, 
    y_pred, 
    ax=ax, 
    cmap="Blues",              
    xticks_rotation="vertical",
    colorbar=True,
    values_format="d"          
)

plt.title("Confusion Matrix – UNSW-NB15 Classification", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Predicted Attack Category", fontsize=12, labelpad=10)
plt.ylabel("True Attack Category", fontsize=12, labelpad=10)
plt.grid(False)               
plt.tight_layout()

plt.savefig("confusion_matrix_plot2.png", dpi=300, bbox_inches="tight")

plt.show()
